# Word embeddings from scratch

Count which words sit near each word, keep the surprising ones, squeeze each word's row to eight numbers, and compare words by the angle between them.

In [1]:
TEXT = """
The cat slept on the sofa all afternoon.
The dog slept by the door all night.
My cat chased a ball across the kitchen.
Our dog chased a ball across the garden.
I fed the cat before work.
I fed the dog after work.
The cat sat on my lap and purred.
The dog sat by my feet and waited.
The vet said the cat was healthy.
The vet said the dog was healthy.
The cat drank water from a bowl.
The dog drank water from a bowl.
We took the dog for a walk in the park.
The cat watched the birds from the window.
I drove the car to work this morning.
She drove the bus to the station.
We parked the car on the street.
They parked the bus by the station.
The car stopped at the red light.
The bus stopped at the red light.
The car was stuck in traffic for an hour.
The bus was stuck in traffic for an hour.
He washed the car on Sunday.
The bus was full of people going to work.
We took the bus to the park.
I filled the car with fuel on the way home.
The car needs new tyres and a new battery.
The bus needs new tyres and a new driver.
The car turned left at the corner.
The bus turned right at the bridge.
A car honked at the bus on the busy road.
The road was closed, so the bus took a long way round.
I ordered a cup of tea at the cafe.
She ordered a cup of coffee at the cafe.
The tea was hot, so I waited a minute.
The tea was cold, so I asked for another.
The coffee was hot and strong.
The coffee was cold and bitter.
The soup was hot and tasted of garlic.
The soup was cold by the time it arrived.
I like my coffee hot in the morning.
I like my coffee cold in the summer.
She poured a cup of hot tea.
He poured a glass of cold water.
The cafe sells hot soup in the winter.
The cafe sells cold drinks in the summer.
A cup of warm milk helps me sleep.
The milk was warm, so I drank it slowly.
The bread was still warm from the oven.
The water was warm enough to swim.
The weather was hot all week.
The weather was cold all week.
It was a hot day, so we stayed inside.
It was a cold day, so we stayed inside.
The wind was cold and the sky was grey.
The sun was hot and the sky was blue.
Summer days are long and hot.
Winter days are short and cold.
The room was warm and quiet.
The evening was warm and calm.
The water in the lake was cold.
The sand on the beach was hot.
We sat in the cafe and watched the rain.
The cafe was busy on Saturday morning.
I paid for the coffee and the soup.
She paid for the tea and the bread.
The cat and the dog slept in the warm room.
We walked to the cafe because the car was at home.
"""

## Count the company

For every word, count the words within two places of it, never across a sentence end.

In [2]:
import re
import numpy as np

sentences = [re.findall(r"[a-z]+", s.lower()) for s in TEXT.strip().splitlines()]
vocab = list(dict.fromkeys(w for s in sentences for w in s))
idx = {w: i for i, w in enumerate(vocab)}

counts = np.zeros((len(vocab), len(vocab)))
for s in sentences:
    for i, w in enumerate(s):
        for j in range(max(0, i - 2), min(len(s), i + 3)):
            if j != i:
                counts[idx[w], idx[s[j]]] += 1

print(len(sentences), "sentences,", len(vocab), "different words")
row = counts[idx["cat"]]
print("near cat:", {vocab[i]: int(row[i]) for i in np.argsort(-row) if row[i] > 0})

68 sentences, 156 different words
near cat: {'the': 9, 'on': 2, 'slept': 1, 'my': 1, 'chased': 1, 'a': 1, 'work': 1, 'before': 1, 'sat': 1, 'fed': 1, 'said': 1, 'and': 1, 'water': 1, 'drank': 1, 'healthy': 1, 'was': 1, 'watched': 1}


## Cosine similarity

1 = the two rows point the same way, 0 = nothing in common.

In [3]:
def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def similarity(table, a, b):
    return round(float(cosine(table[idx[a]], table[idx[b]])), 2)

for a, b in [("cat", "dog"), ("cat", "car")]:
    print(a, b, similarity(counts, a, b))

cat dog 0.88
cat car 0.84


## Positive PMI

log(how many times more often than luck two words meet). Below luck it is negative; keep only the positive scores.

In [4]:
total = counts.sum()
row_sums, col_sums = counts.sum(1, keepdims=True), counts.sum(0, keepdims=True)
with np.errstate(divide="ignore"):
    pmi = np.log(counts * total / (row_sums * col_sums))   # counts / luck, luck = row_sums * col_sums / total
ppmi = np.where(pmi > 0, pmi, 0.0)

for a, b in [("cat", "dog"), ("cat", "car")]:
    print(a, b, similarity(ppmi, a, b))

cat dog 0.61
cat car 0.07


## SVD: eight numbers per word

In [5]:
U, S, Vt = np.linalg.svd(ppmi, full_matrices=False)
emb = U[:, :8] * S[:8]

print("cat:", np.round(emb[idx["cat"]], 2))
for a, b in [("cat", "dog"), ("cat", "car"), ("before", "after")]:
    print(a, b, similarity(emb, a, b))

cat: [ 2.35  1.53 -0.17  0.79  1.61 -0.74  1.63  1.75]
cat dog 0.97
cat car 0.23
before after 1.0


## Closest to hot

In [6]:
def closest(word, k=5):
    v = emb[idx[word]]
    scores = {w: float(cosine(v, emb[i])) for w, i in idx.items() if w != word}
    return [(w, round(s, 2)) for w, s in sorted(scores.items(), key=lambda t: -t[1])[:k]]

print(closest("hot"))
print("hot cold", similarity(emb, "hot", "cold"), "| hot warm", similarity(emb, "hot", "warm"))

[('cold', 0.99), ('weather', 0.99), ('soup', 0.98), ('wind', 0.98), ('sun', 0.98)]
hot cold 0.99 | hot warm 0.37


## PCA: a flat map

Scale each word's eight numbers to length 1, centre them, and keep the two directions they spread out along the most.

In [7]:
words = ["cat", "dog", "car", "bus", "tea", "coffee", "soup", "milk", "hot", "cold", "warm"]
X = emb[[idx[w] for w in words]]
X = X / np.linalg.norm(X, axis=1, keepdims=True)
X = X - X.mean(0)
_, _, V = np.linalg.svd(X, full_matrices=False)
xy = X @ V[:2].T
for w, (x, y) in zip(words, xy):
    print(f"{w:7} {x:6.2f} {y:6.2f}")

cat      -0.77   0.05
dog      -0.73   0.07
car      -0.52  -0.58
bus      -0.28  -0.57
tea       0.45  -0.15
coffee    0.35  -0.13
soup      0.55  -0.08
milk     -0.16   0.78
hot       0.58  -0.06
cold      0.56  -0.04
warm     -0.03   0.72


## Try your own

Add sentences to TEXT, run all the cells again, and ask for the closest words to any word in it.

In [8]:
print(closest("dog"))

[('cat', 0.97), ('said', 0.96), ('healthy', 0.95), ('our', 0.95), ('fed', 0.94)]
